# NB04 · Molecular Graphs and Matrices
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prateekpaul/Cheminformatics/blob/main/solutions/nb04_molecular_graphs_and_matrices.ipynb)

**Estimated time:** ~75 min | **Difficulty:** ●●○

A rigorous treatment of molecular graph theory, converting chemical structures to NetworkX graph objects, constructing adjacency, bond-order, and topological distance matrices, calculating walks, paths, and classical topological indices (Wiener, Zagreb, Randić), evaluating subgraph isomorphism algorithms (VF2), and clarifying complexity theory boundaries.

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Represent** molecules as colored, labeled undirected graphs using `networkx` and RDKit.
2. **Construct and interpret** core molecular matrices: adjacency matrix ($A$), bond-order matrix ($B$), topological distance matrix ($D$), and connection tables.
3. **Compute** molecular walk counts ($MWC_k$) and self-returning walk counts ($SRW_k = \text{Tr}(A^k)$), validating fundamental graph theorems ($MWC_0 = V$, $MWC_1 = 2E$).
4. **Calculate** classic topological indices (Wiener Index, Zagreb $M_1/M_2$, Randić branching index) and explain their sensitivity to molecular branching.
5. **Implement** graph and subgraph isomorphism matching using NetworkX VF2 and RDKit substructure matching.
6. **Differentiate** computational complexity classes: subgraph isomorphism is NP-complete, while general graph isomorphism is NP-intermediate.

## Prerequisites and Links

- NB00, NB01, NB03 completed.
- Lecture 02 (Graph theory, matrices, InChI, isomorphism) and Lecture 05 (Molecular graph, walks, paths, topological distances).

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger

SEED = 42
FULL_RUN = False
random.seed(SEED)
np.random.seed(SEED)
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("NB04 setup completed.")

## 1. Molecules as Attributed Graphs

In chemical graph theory, a molecule is formalized as an undirected attributed graph $G = (V, E)$:
- **Vertices ($V$):** Heavy atoms, decorated with attributes (element, atomic number, formal charge, degree, hybridization, aromaticity).
- **Edges ($E$):** Covalent bonds, decorated with bond type (single, double, triple, aromatic) and bond order ($1.0, 2.0, 3.0, 1.5$).

### 'Graphs are Purely About Connectivity'

A graph has no intrinsic 2D or 3D coordinates. Different visual depictions or conformers of the same molecule correspond to the exact same graph topology.

In [ ]:
from chemlib import graphs

mol_aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
G_aspirin = graphs.mol_to_networkx(mol_aspirin)

print(f"Aspirin heavy atoms (nodes): {G_aspirin.number_of_nodes()}")
print(f"Aspirin covalent bonds (edges): {G_aspirin.number_of_edges()}")

# Inspect attributes of node 0 and edge (0, 1)
print(f"Node 0 attributes: {G_aspirin.nodes[0]}")
sample_edge = list(G_aspirin.edges(data=True))[0]
print(f"Sample edge attributes: {sample_edge}")

## 2. Molecular Matrices for Aspirin

A graph $G$ with $N$ vertices can be represented through several fundamental $N \times N$ matrices:

1. **Adjacency Matrix ($A$):**
   $$A_{ij} = \begin{cases} 1 & \text{if atom } i \text{ and } j \text{ are bonded} \\ 0 & \text{otherwise} \end{cases}$$
2. **Bond-Order Matrix ($B$):**
   $$B_{ij} = \begin{cases} b_{ij} & \text{bond order between } i \text{ and } j \\ 0 & \text{otherwise} \end{cases}$$
3. **Topological Distance Matrix ($D$):**
   $$D_{ij} = \text{shortest path distance (number of bonds) between atom } i \text{ and } j$$

In [ ]:
A_mat = graphs.get_adjacency_matrix(mol_aspirin)
B_mat = graphs.get_bond_order_matrix(mol_aspirin)
D_mat = graphs.get_distance_matrix(mol_aspirin)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

im0 = axes[0].imshow(A_mat, cmap="Blues", interpolation="nearest")
axes[0].set_title("Adjacency Matrix ($A$)", fontsize=11)
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(B_mat, cmap="viridis", interpolation="nearest")
axes[1].set_title("Bond-Order Matrix ($B$)", fontsize=11)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(D_mat, cmap="magma", interpolation="nearest")
axes[2].set_title("Distance Matrix ($D$)", fontsize=11)
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 3. Walks, Paths, Cycles and Matrix Powers

- **Walk:** An alternating sequence of vertices and edges $v_0, e_1, v_1, \dots, e_k, v_k$ where vertices and edges can be repeated.
- **Path:** A walk in which all vertices (and thus edges) are distinct.
- **Self-Returning Walk:** A walk that starts and ends at the same vertex ($v_0 = v_k$).

### The Walk Counting Theorem

A cornerstone theorem of algebraic graph theory states that the entry $(A^k)_{ij}$ of the $k$-th power of the adjacency matrix equals the **exact number of walks of length $k$** between vertex $i$ and vertex $j$.

Consequently:
- **Total Molecular Walk Count ($MWC_k$):**
  $$MWC_k = \sum_{i=1}^N \sum_{j=1}^N (A^k)_{ij}$$
  - For $k = 0$: $A^0 = I \implies MWC_0 = \text{Tr}(I) = |V|$ (number of heavy atoms).
  - For $k = 1$: $A^1 = A \implies MWC_1 = \sum_{i,j} A_{ij} = 2 |E|$ (twice the number of bonds).
- **Self-Returning Walk Count ($SRW_k$):**
  $$SRW_k = \text{Tr}(A^k) = \sum_{i=1}^N (A^k)_{ii}$$
  - $SRW_0 = |V|$
  - $SRW_1 = 0$ (simple graphs have no self-loops)
  - $SRW_2 = \sum \text{deg}(v_i) = 2 |E|$
  - $SRW_3 = 6 \times (\text{number of triangles/3-cycles})$

In [ ]:
n_atoms = mol_aspirin.GetNumHeavyAtoms()
n_bonds = mol_aspirin.GetNumBonds()

mwc_0 = graphs.molecular_walk_count(mol_aspirin, 0)
mwc_1 = graphs.molecular_walk_count(mol_aspirin, 1)
mwc_2 = graphs.molecular_walk_count(mol_aspirin, 2)
mwc_3 = graphs.molecular_walk_count(mol_aspirin, 3)

srw_0 = graphs.self_returning_walk_count(mol_aspirin, 0)
srw_1 = graphs.self_returning_walk_count(mol_aspirin, 1)
srw_2 = graphs.self_returning_walk_count(mol_aspirin, 2)
srw_3 = graphs.self_returning_walk_count(mol_aspirin, 3)

print(f"Number of heavy atoms: {n_atoms}, Number of bonds: {n_bonds}")
print(f"MWC_0: {mwc_0} (Expected: {n_atoms})")
print(f"MWC_1: {mwc_1} (Expected: {2 * n_bonds})")
print(f"MWC_2: {mwc_2}, MWC_3: {mwc_3}")
print(f"SRW_0: {srw_0}, SRW_1: {srw_1}, SRW_2: {srw_2}, SRW_3: {srw_3}")

assert mwc_0 == n_atoms
assert mwc_1 == 2 * n_bonds
assert srw_0 == n_atoms
assert srw_1 == 0
assert srw_2 == 2 * n_bonds
print("✔ Walk counting theorems verified on aspirin!")

## 4. Classic Topological Indices

Topological indices compress the molecular graph into numerical scalar invariants correlated with physicochemical properties:

1. **Wiener Index ($W$, 1947):** Half-sum of all entries in the topological distance matrix:
   $$W = \frac{1}{2} \sum_{i=1}^N \sum_{j=1}^N D_{ij}$$
   Historically used by Harold Wiener to predict the boiling points of paraffin alkanes.
2. **First Zagreb Index ($M_1$):**
   $$M_1 = \sum_{i=1}^N (\delta_i)^2$$
   where $\delta_i$ is the vertex degree of atom $i$.
3. **Second Zagreb Index ($M_2$):**
   $$M_2 = \sum_{(i,j) \in E} \delta_i \delta_j$$
4. **Randić Connectivity Index ($\chi$, 1975):**
   $$\chi = \sum_{(i,j) \in E} \frac{1}{\sqrt{\delta_i \delta_j}}$$
   Measures molecular branching and surface area.

In [ ]:
pentanes = [
    ("n-Pentane", "CCCCC"),
    ("Isopentane", "CCC(C)C"),
    ("Neopentane", "CC(C)(C)C")
]

topo_results = []
for name, smi in pentanes:
    m = Chem.MolFromSmiles(smi)
    w = graphs.wiener_index(m)
    m1 = graphs.zagreb_m1_index(m)
    m2 = graphs.zagreb_m2_index(m)
    chi = round(graphs.randic_index(m), 4)
    topo_results.append({
        "Isomer": name,
        "Wiener Index (W)": w,
        "Zagreb M1": m1,
        "Zagreb M2": m2,
        "Randic Index": chi
    })

df_topo = pd.DataFrame(topo_results)
print(df_topo)

> [!NOTE] Branching Effect on Topological Indices
> Notice how increasing branching from n-pentane $\to$ isopentane $\to$ neopentane strictly **decreases the Wiener index** ($20 \to 18 \to 16$) because atoms become topologically closer to each other.

## 5. Graph and Subgraph Isomorphism

- **Graph Isomorphism:** Determines whether two entire molecular graphs $G_1$ and $G_2$ have identical topological connectivity ($G_1 \cong G_2$).
- **Subgraph Isomorphism:** Determines whether a query pattern graph $Q$ is isomorphic to a subgraph of target $T$ ($Q \subseteq T$). This is the foundation of chemical **substructure searching**.

### The VF2 Algorithm

NetworkX implements the **VF2 algorithm** (Cordella et al., 2001), which prunes search trees using feasibility rules matching vertex elements and edge bond orders.

In [ ]:
benzene = Chem.MolFromSmiles("c1ccccc1")

# Subgraph test: Benzene in Aspirin
has_subgraph_vf2 = graphs.check_subgraph_isomorphism_vf2(mol_aspirin, benzene)
has_subgraph_rdkit = mol_aspirin.HasSubstructMatch(benzene)

print(f"VF2 Subgraph Match (Benzene in Aspirin):   {has_subgraph_vf2}")
print(f"RDKit Subgraph Match (Benzene in Aspirin): {has_subgraph_rdkit}")
assert has_subgraph_vf2 is True and has_subgraph_rdkit is True

## 6. Computational Complexity: Subgraph vs. Graph Isomorphism

> [!IMPORTANT] Note vs. slides (Errata E6 / E7)
> Lecture 06 casually refers to graph isomorphism as "NP-complete". We must make the precise theoretical distinction:
>
> 1. **Subgraph Isomorphism is NP-complete.** Proved by Stephen Cook (1971) and Richard Karp (1972). In the worst case, deciding whether $Q \subseteq T$ scales exponentially with query size.
> 2. **General Graph Isomorphism is NOT known to be NP-complete.** It is widely believed to occupy the rare **NP-intermediate** complexity class. In 2015, László Babai announced a **quasi-polynomial time algorithm** solving graph isomorphism in $2^{O((\log n)^c)}$ time!
>
> In cheminformatics database search, because subgraph isomorphism is NP-complete, chemical databases never run VF2 directly on millions of molecules. Instead, they first apply fast **fingerprint screening** to filter out 99% of non-matching candidates before running exact subgraph isomorphism on the remainder.

### Exercise 4.1

Implement `adjacency_from_smiles(smiles: str) -> np.ndarray` from scratch without using `Chem.GetAdjacencyMatrix`. Iterate over `mol.GetBonds()`, extracting begin and end atom indices to populate an $N \times N$ binary symmetric numpy matrix.

In [ ]:
def adjacency_from_smiles(smiles: str) -> np.ndarray:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    n = mol.GetNumHeavyAtoms()
    adj = np.zeros((n, n), dtype=int)
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        adj[i, j] = 1
        adj[j, i] = 1
    return adj

In [ ]:
# Self-check for Exercise 4.1
test_mol = "CC(=O)Oc1ccccc1C(=O)O"
a_custom = adjacency_from_smiles(test_mol)
a_rdkit = Chem.GetAdjacencyMatrix(Chem.MolFromSmiles(test_mol))
assert np.array_equal(a_custom, a_rdkit)
print('✔ Exercise 4.1 passed!')

### Exercise 4.2

Prove that pure molecular graphs discard stereochemistry: construct NetworkX graphs for $(R)$-lactic acid (`C[C@@H](O)C(=O)O`) and $(S)$-lactic acid (`C[C@H](O)C(=O)O`). Write a function `are_stereoisomers_graph_isomorphic(smi1: str, smi2: str) -> bool` and show that `nx.is_isomorphic` returns `True`.

In [ ]:
def are_stereoisomers_graph_isomorphic(smi1: str, smi2: str) -> bool:
    m1 = Chem.MolFromSmiles(smi1)
    m2 = Chem.MolFromSmiles(smi2)
    G1 = graphs.mol_to_networkx(m1)
    G2 = graphs.mol_to_networkx(m2)
    return nx.is_isomorphic(G1, G2, node_match=lambda a, b: a["element"] == b["element"])

In [ ]:
# Self-check for Exercise 4.2
s_lactic = "C[C@H](O)C(=O)O"
r_lactic = "C[C@@H](O)C(=O)O"
assert are_stereoisomers_graph_isomorphic(s_lactic, r_lactic) is True
print('✔ Exercise 4.2 passed!')

### Exercise 4.3

Write a function `count_molecular_rings(smiles: str) -> int` that converts a molecule into a NetworkX graph and computes the number of fundamental rings using `nx.cycle_basis(G)`. Verify against RDKit's `mol.GetRingInfo().NumRings()` on hexane (0), benzene (1), naphthalene (2), and anthracene (3).

In [ ]:
def count_molecular_rings(smiles: str) -> int:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    G = graphs.mol_to_networkx(mol)
    cycles = nx.cycle_basis(G)
    return len(cycles)

In [ ]:
# Self-check for Exercise 4.3
assert count_molecular_rings("CCCCCC") == 0       # n-hexane
assert count_molecular_rings("c1ccccc1") == 1      # benzene
assert count_molecular_rings("c1ccc2ccccc2c1") == 2 # naphthalene
assert count_molecular_rings("c1ccc2cc3ccccc3cc2c1") == 3 # anthracene
print('✔ Exercise 4.3 passed!')

## Common Pitfalls

1. **Aromatic vs. Alternating Single/Double Bonds:** In an unweighted adjacency matrix, both single and double bonds are encoded as binary `1`. Use the bond-order matrix or attributed edges if bond orders must be distinguished.
2. **Automorphisms and Multiple Subgraph Mappings:** Symmetric molecules (e.g. benzene with 12 automorphisms) yield multiple isomorphic mappings for a single substructure query.
3. **Hydrogen Suppression in Graph Algorithms:** Cycle basis algorithms and Wiener index computations assume hydrogen-depleted graphs. Adding explicit hydrogens dramatically alters vertex degrees and shortest paths.

## Summary and Key Takeaways

- Molecules can be formalized as colored, labeled undirected graphs $G = (V, E)$.
- Graph connectivity is invariant to visual 2D diagram layouts and 3D conformations.
- The adjacency matrix power $(A^k)_{ij}$ gives the exact number of walks of length $k$ between vertices $i$ and $j$.
- Molecular walk counts verify fundamental theorems: $MWC_0 = V$ and $MWC_1 = 2E$.
- Topological indices (Wiener, Zagreb, Randić) encode molecular branching and size into topological scalars.
- Subgraph isomorphism is NP-complete, necessitating fingerprint screening filters in database search.

## References

- Wiener, H. Structural determination of paraffin boiling points. *J. Am. Chem. Soc.* **1947**, 69, 17–20.
- Randić, M. On characterization of molecular branching. *J. Am. Chem. Soc.* **1975**, 97, 6609–6615.
- Cordella, L. P. et al. An improved algorithm for matching large graphs (VF2). *IEEE TPAMI* **2004**, 26, 1367–1372.
- Babai, L. Graph isomorphism in quasipolynomial time. *arXiv:1512.03547*, 2015.

In [ ]:
# Self-test check
print("✔ NB04 checks passed")